# Scaling Laws for Training Budget

> Goal: estimate compute-optimal model size and training tokens for a fixed budget (example: **$1000**).

We use a simplified Chinchilla-style setup:
- Training FLOPs: $C \approx 6 N D$
- Compute-optimal token ratio: $D \approx 20N$

Where:
- $N$ = number of model parameters
- $D$ = number of training tokens
- $C$ = total training FLOPs

From these assumptions:
- $N_{opt} = \sqrt{C/120}$
- $D_{opt} = 20N_{opt}$

This notebook also estimates checkpoint size from parameter count and dtype.

In [2]:
import math
from dataclasses import dataclass
import pandas as pd

@dataclass
class BudgetAssumptions:
    budget_usd: float = 1000.0
    gpu_hourly_usd: float = 2.5   # e.g. cloud A100 80GB spot/on-demand blend
    peak_tflops_bf16: float = 312.0
    utilization: float = 0.35     # real training efficiency vs peak

def total_train_flops(a: BudgetAssumptions) -> float:
    gpu_hours = a.budget_usd / a.gpu_hourly_usd
    eff_flops_per_sec = a.peak_tflops_bf16 * 1e12 * a.utilization
    return gpu_hours * 3600 * eff_flops_per_sec

def chinchilla_optimal_from_compute(C: float):
    # C ~= 6ND, D ~= 20N -> C ~= 120N^2
    n_params = math.sqrt(C / 120.0)
    d_tokens = 20.0 * n_params
    return n_params, d_tokens

def checkpoint_size_mb(n_params: float, dtype: str = 'fp32') -> float:
    bytes_per_param = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1}[dtype]
    return n_params * bytes_per_param / (1024**2)

def human(n):
    if n >= 1e12:
        return f"{n/1e12:.2f}T"
    if n >= 1e9:
        return f"{n/1e9:.2f}B"
    if n >= 1e6:
        return f"{n/1e6:.2f}M"
    if n >= 1e3:
        return f"{n/1e3:.2f}K"
    return f"{n:.0f}"

In [3]:
# Baseline scenario: $1000 budget
assump = BudgetAssumptions(
    budget_usd=1000.0,
    gpu_hourly_usd=2.5,
    peak_tflops_bf16=312.0,
    utilization=0.35,
)

C = total_train_flops(assump)
N_opt, D_opt = chinchilla_optimal_from_compute(C)

summary = pd.DataFrame([
    {
        'budget_usd': assump.budget_usd,
        'gpu_hourly_usd': assump.gpu_hourly_usd,
        'peak_tflops_bf16': assump.peak_tflops_bf16,
        'utilization': assump.utilization,
        'total_train_flops': C,
        'optimal_params': N_opt,
        'optimal_tokens': D_opt,
        'ckpt_fp32_mb': checkpoint_size_mb(N_opt, 'fp32'),
        'ckpt_fp16_mb': checkpoint_size_mb(N_opt, 'fp16'),
    }
])

display(summary.style.format({
    'budget_usd': '{:,.0f}',
    'gpu_hourly_usd': '{:,.2f}',
    'peak_tflops_bf16': '{:,.0f}',
    'utilization': '{:.0%}',
    'total_train_flops': '{:.3e}',
    'optimal_params': '{:.3e}',
    'optimal_tokens': '{:.3e}',
    'ckpt_fp32_mb': '{:,.1f}',
    'ckpt_fp16_mb': '{:,.1f}',
}))

print(f"Compute-optimal model size: {human(N_opt)} parameters")
print(f"Compute-optimal token budget: {human(D_opt)} tokens")

,budget_usd,gpu_hourly_usd,peak_tflops_bf16,utilization,total_train_flops,optimal_params,optimal_tokens,ckpt_fp32_mb,ckpt_fp16_mb
0,"1,000",2.50,312,35%,1.572e+20,1.145e+09,2.289e+10,"4,366.8","2,183.4"


Compute-optimal model size: 1.14B parameters
Compute-optimal token budget: 22.89B tokens


In [ ]:
# Compare several hardware/cost assumptions for the same $1000 budget
scenarios = [
    ('A100 cloud', 2.5, 312.0, 0.35),
    ('H100 cloud', 4.5, 989.0, 0.30),
    ('Consumer 4090-like', 1.0, 165.0, 0.35),
]

rows = []
for name, usd_h, tflops, util in scenarios:
    a = BudgetAssumptions(
        budget_usd=1000.0,
        gpu_hourly_usd=usd_h,
        peak_tflops_bf16=tflops,
        utilization=util,
    )
    C = total_train_flops(a)
    N_opt, D_opt = chinchilla_optimal_from_compute(C)
    rows.append({
        'scenario': name,
        'gpu_hours': a.budget_usd / a.gpu_hourly_usd,
        'effective_tflops': tflops * util,
        'optimal_params_readable': human(N_opt),
        'optimal_tokens_readable': human(D_opt),
        'optimal_params': N_opt,
        'optimal_tokens': D_opt,
    })

cmp_df = pd.DataFrame(rows).sort_values('optimal_params', ascending=False)
display(
    cmp_df[
        [
            'scenario',
            'gpu_hours',
            'effective_tflops',
            'optimal_params_readable',
            'optimal_tokens_readable',
        ]
    ]
)

,scenario,gpu_hours,effective_tflops,optimal_params_readable,optimal_tokens_readable
1,H100 cloud,222.222222,296.70,1.41B,28.13B
2,Consumer 4090-like,1000.000000,57.75,1.32B,26.32B
0,A100 cloud,400.000000,109.20,1.14B,22.89B


In [5]:
# Compare with current Krasnal model (from current config: ~11.77M params)
current_params = 11_771_904

# Always use the baseline assumptions from the baseline cell
C_baseline = total_train_flops(assump)
N_opt_baseline, D_opt_baseline = chinchilla_optimal_from_compute(C_baseline)

# With the same compute budget, how many tokens could we train this model on?
D_for_current = C_baseline / (6.0 * current_params)

compare_current = pd.DataFrame([
    {
        'model': 'Current Krasnal config',
        'params': current_params,
        'tokens_for_budget': D_for_current,
        'ckpt_fp32_mb': checkpoint_size_mb(current_params, 'fp32'),
        'ckpt_fp16_mb': checkpoint_size_mb(current_params, 'fp16'),
    },
    {
        'model': 'Compute-optimal (Chinchilla approx)',
        'params': N_opt_baseline,
        'tokens_for_budget': D_opt_baseline,
        'ckpt_fp32_mb': checkpoint_size_mb(N_opt_baseline, 'fp32'),
        'ckpt_fp16_mb': checkpoint_size_mb(N_opt_baseline, 'fp16'),
    },
])

display(compare_current.style.format({
    'params': '{:.3e}',
    'tokens_for_budget': '{:.3e}',
    'ckpt_fp32_mb': '{:,.1f}',
    'ckpt_fp16_mb': '{:,.1f}',
}))

print(f"Current model checkpoint size (fp32): {checkpoint_size_mb(current_params, 'fp32'):.1f} MB")
print(f"Current model checkpoint size (fp16): {checkpoint_size_mb(current_params, 'fp16'):.1f} MB")

,model,params,tokens_for_budget,ckpt_fp32_mb,ckpt_fp16_mb
0,Current Krasnal config,1.177e+07,2.226e+12,44.9,22.5
1,Compute-optimal (Chinchilla approx),1.145e+09,2.289e+10,"4,366.8","2,183.4"


Current model checkpoint size (fp32): 44.9 MB
Current model checkpoint size (fp16): 22.5 MB


## Empirical local benchmark (short training run)

Because the local parquet dataset is missing, this section runs a **real optimizer training loop** on synthetic token batches with the same model shape.  
This still gives a practical estimate of local throughput (tokens/s, steps/s), which we can convert to budget-based scaling estimates.

In [ ]:
import time
import sys
import importlib
from pathlib import Path

import torch

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

model_module = importlib.import_module('model')
config_module = importlib.import_module('config')
GPT = model_module.GPT
GPTConfig = model_module.GPTConfig
ChessGPTConfig = config_module.ChessGPTConfig

bench_steps = 80
warmup_steps = 10
target_seconds = 30
bench_batch_size = 16

mconf = ChessGPTConfig()
model_conf = GPTConfig(
    block_size=mconf.block_size,
    vocab_size=1971,
    n_layer=mconf.n_layer,
    n_head=mconf.n_head,
    n_embd=mconf.n_embd,
    dropout=mconf.dropout,
    bias=mconf.bias,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float32

torch.manual_seed(42)
model = GPT(model_conf).to(device)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

B = bench_batch_size
T = model_conf.block_size
vocab_size = model_conf.vocab_size

times = []
losses = []
start_wall = time.perf_counter()

for step in range(bench_steps):
    x = torch.randint(0, vocab_size, (B, T), device=device)
    y = torch.roll(x, shifts=-1, dims=1)

    t0 = time.perf_counter()
    with torch.autocast(
        device_type='cuda',
        dtype=torch.bfloat16,
        enabled=(device == 'cuda' and dtype == torch.bfloat16),
    ):
        _, loss = model(x, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    t1 = time.perf_counter()

    losses.append(float(loss.item()))
    if step >= warmup_steps:
        times.append(t1 - t0)

    if (time.perf_counter() - start_wall) > target_seconds and step >= warmup_steps + 5:
        break

mean_step_s = sum(times) / len(times)
steps_per_s = 1.0 / mean_step_s
tokens_per_step = B * T
tokens_per_s = tokens_per_step * steps_per_s

# compute rate proxy from C ~= 6*N*D
N_current = sum(p.numel() for p in model.parameters())
flops_per_s_est = 6.0 * N_current * tokens_per_s

bench_stats = {
    'device': device,
    'dtype': str(dtype).replace('torch.', ''),
    'steps_ran': warmup_steps + len(times),
    'batch_size': B,
    'seq_len': T,
    'params': N_current,
    'mean_step_seconds': mean_step_s,
    'steps_per_second': steps_per_s,
    'tokens_per_second': tokens_per_s,
    'flops_per_second_est': flops_per_s_est,
    'last_loss': losses[-1],
}

bench_df = pd.DataFrame([bench_stats])
display(bench_df.style.format({
    'mean_step_seconds': '{:.4f}',
    'steps_per_second': '{:.3f}',
    'tokens_per_second': '{:,.0f}',
    'params': '{:,.0f}',
    'flops_per_second_est': '{:.3e}',
    'last_loss': '{:.4f}',
}))

Using device: cuda
number of parameters: 11.38M


,device,dtype,steps_ran,batch_size,seq_len,params,mean_step_seconds,steps_per_second,tokens_per_second,flops_per_second_est,last_loss
0,cuda,bfloat16,80,16,1024,"11,771,904",0.0221,45.176,"740,171",5.228e+13,7.5932


In [7]:
# Empirical scaling estimate for a given budget using measured local throughput
local_hourly_usd = 1.0  # put your own effective local GPU cost per hour
budget_usd_empirical = 1000.0

seconds_affordable = (budget_usd_empirical / local_hourly_usd) * 3600.0
C_empirical_budget = flops_per_s_est * seconds_affordable

N_opt_emp, D_opt_emp = chinchilla_optimal_from_compute(C_empirical_budget)
D_current_emp = C_empirical_budget / (6.0 * N_current)

empirical_df = pd.DataFrame([
    {
        'budget_usd': budget_usd_empirical,
        'local_hourly_usd': local_hourly_usd,
        'measured_tokens_per_second': tokens_per_s,
        'estimated_budget_flops': C_empirical_budget,
        'opt_params_from_local_rate': N_opt_emp,
        'opt_tokens_from_local_rate': D_opt_emp,
        'current_model_tokens_for_budget': D_current_emp,
        'current_ckpt_fp32_mb': checkpoint_size_mb(N_current, 'fp32'),
        'opt_ckpt_fp32_mb': checkpoint_size_mb(N_opt_emp, 'fp32'),
    }
])

display(empirical_df.style.format({
    'budget_usd': '{:,.0f}',
    'local_hourly_usd': '{:,.2f}',
    'measured_tokens_per_second': '{:,.0f}',
    'estimated_budget_flops': '{:.3e}',
    'opt_params_from_local_rate': '{:.3e}',
    'opt_tokens_from_local_rate': '{:.3e}',
    'current_model_tokens_for_budget': '{:.3e}',
    'current_ckpt_fp32_mb': '{:,.1f}',
    'opt_ckpt_fp32_mb': '{:,.1f}',
}))

print(f"Empirical local rate: {tokens_per_s:,.0f} tokens/s")
print(f"Empirical-optimal params (@ ${budget_usd_empirical:,.0f}): {human(N_opt_emp)}")
print(f"Empirical-optimal tokens (@ ${budget_usd_empirical:,.0f}): {human(D_opt_emp)}")

,budget_usd,local_hourly_usd,measured_tokens_per_second,estimated_budget_flops,opt_params_from_local_rate,opt_tokens_from_local_rate,current_model_tokens_for_budget,current_ckpt_fp32_mb,opt_ckpt_fp32_mb
0,"1,000",1.00,"740,171",1.882e+20,1.252e+09,2.505e+10,2.665e+12,44.9,"4,777.3"


Empirical local rate: 740,171 tokens/s
Empirical-optimal params (@ $1,000): 1.25B
Empirical-optimal tokens (@ $1,000): 25.05B
